In [3]:
import pandas as pd
import numpy as np
import plotly.express as px

# Import du dataset
L'encodage par défaut a planté, d'où l'encoding en latin-1

In [4]:
df = pd.read_csv("data/tinder_data.csv", encoding="latin-1")
df.head()

,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


# Infos générales
Le dataset contient **8378 lignes** (une par "date" de 4 minutes) et **195 colonnes**. Chaque ligne représente la rencontre entre deux participants avec les ratings donnés, les préférences déclarées en amont et la décision finale.


In [7]:
df.shape

(8378, 195)

In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 195 entries, iid to amb5_3
dtypes: float64(174), int64(13), object(8)
memory usage: 12.5+ MB


# Taux de "oui" et match

In [9]:
df["dec"].value_counts(normalize=True)


dec
0    0.580091
1    0.419909
Name: proportion, dtype: float64

In [10]:
df["match"].value_counts(normalize=True)


match
0    0.835283
1    0.164717
Name: proportion, dtype: float64

# Les participants

**Population étudiée**
- 551 participants uniques
- 21 sessions de speed dating (entre octobre 2002 et avril 2004)
- Répartition équilibrée : 277 hommes (52 %) et 274 femmes (48 %)


In [11]:
# Nombre de participants uniques
df["iid"].nunique()


551

In [12]:
# Nombre de waves (sessions de speed dating)
df["wave"].nunique()


21

In [14]:
# Répartition par genre (0 = femme, 1 = homme)
df.drop_duplicates("iid")["gender"].value_counts()


gender
1    277
0    274
Name: count, dtype: int64

# Valeurs manquantes

**Valeurs manquantes sur les variables clés**

Les cibles (`dec`, `match`) et variables structurelles (`gender`, `order`, `samerace`) sont complètes. Les ratings post-date ont quelques centaines de manquants (participants qui n'ont pas noté tous leurs partenaires sur tous les critères). Les deux colonnes les plus incomplètes sont `shar` (intérêts partagés, 13 % de NaN) et `amb` (ambition, 8,5 %)


In [15]:
# Colonnes qu'on va utiliser dans nos analyses
colonnes_cles = [
    "dec", "match", "gender",              # cibles et genre
    "attr", "sinc", "intel", "fun", "amb", "shar",   # ratings post-date
    "like", "prob",                                   # appréciation globale
    "int_corr", "samerace", "age", "age_o", "order"   # contexte
]

df[colonnes_cles].isna().sum()


dec            0
match          0
gender         0
attr         202
sinc         277
intel        296
fun          350
amb          712
shar        1067
like         240
prob         309
int_corr     158
samerace       0
age           95
age_o        104
order          0
dtype: int64

# Taux de "oui" par genre

les hommes sont plus facilement enclins à vouloir revoir une partenaire, les femmes sont plus sélectives.

In [16]:
# Taux moyen de "oui" par genre
df.groupby("gender")["dec"].mean()


gender
0    0.365440
1    0.474249
Name: dec, dtype: float64

In [17]:
# Préparer les données pour le graphique
taux_par_genre = df.groupby("gender")["dec"].mean().reset_index()
taux_par_genre["gender"] = taux_par_genre["gender"].map({0: "Femmes", 1: "Hommes"})

fig = px.bar(
    taux_par_genre,
    x="gender",
    y="dec",
    title="Taux de 'oui' par genre",
    labels={"gender": "Genre", "dec": "Taux de décisions positives"},
    text_auto=".1%"
)
fig.show()


# Quels attributs distinguent un oui d'un non

In [18]:
attributs = ["attr", "sinc", "intel", "fun", "amb", "shar"]

# Moyenne de chaque rating selon genre et décision
moyennes = df.groupby(["gender", "dec"])[attributs].mean().round(2)
moyennes


attr  sinc  intel   fun   amb  shar
gender dec                                     
0      0    5.23  6.78   7.16  5.61  6.70  4.70
       1    7.07  7.63   7.93  7.38  7.37  6.58
1      0    5.53  6.95   6.98  5.80  6.24  4.75
       1    7.45  7.57   7.62  7.30  7.00  6.37

-> Qui font dire oui que ça soit pour un homme ou pour une femme : **attractivité, intérêts partagés, fun** . L'intelligence, la sincérité et l'ambition sont notées élevées dans tous les cas et ne pèse pas vraiment sur la decision.

# Ecarts "oui / non "

In [21]:
# On calcule l'écart entre la note moyenne quand "oui" et quand "non", pour chaque genre et chaque attribut
ecarts = (
    df.groupby(["gender", "dec"])[attributs]
    .mean()
    .unstack("dec")          # met 'dec' en colonnes
    .swaplevel(axis=1)       # (attribut, dec) -> (dec, attribut)
)

# Écart = moyenne quand dec=1 moins moyenne quand dec=0
ecart_final = (ecarts[1] - ecarts[0]).round(2)
ecart_final.index = ecart_final.index.map({0: "Femmes", 1: "Hommes"})
ecart_final


,attr,sinc,intel,fun,amb,shar
gender,,,,,,
Femmes,1.84,0.85,0.77,1.77,0.67,1.89
Hommes,1.92,0.62,0.64,1.50,0.76,1.62


In [22]:

ecart_long = ecart_final.reset_index().melt(
    id_vars="gender",
    var_name="Attribut",
    value_name="Écart oui − non"
)
ecart_long = ecart_long.rename(columns={"gender": "Genre"})

fig = px.bar(
    ecart_long,
    x="Attribut",
    y="Écart oui − non",
    color="Genre",
    barmode="group",
    title="Ce qui fait dire 'oui' : écart entre la note moyenne donnée quand on accepte vs quand on refuse",
    text_auto=".2f"
)
fig.show()


**Ce qui déclenche un "oui", pour les deux genres**

Les trois attributs qui différencient le plus une décision positive d'une décision négative sont, dans l'ordre : l'**attractivité** (+1,8 à +1,9), les **intérêts partagés** (+1,6 à +1,9) et le **fun** (+1,5 à +1,8).

À l'inverse, la **sincérité**, l'**intelligence** et l'**ambition** ne discriminent presque pas la décision : ces notes sont déjà élevées (6-7/10) même quand la personne dit "non". Ce sont des prérequis attendus, pas des déclencheurs.

**Différence entre genres :**
- Les **hommes** sont surtout guidés par l'**attractivité** (écart +1,92, le plus élevé).
- Les **femmes** sont plus équilibrées : attractivité, intérêts partagés et fun comptent presque également (+1,77 à +1,88), avec même un léger avantage pour les intérêts partagés.

→ Résumé: pour les hommes, l'apparence reste le levier numéro un. Pour les femmes, mettre en avant les centres d'intérêt a autant de poids que la photo.


# Corrélation entre chaque attribut et la décision, par genre

In [23]:
# Pour chaque genre, corrélation entre 'dec' et chacun des 6 attributs
correlations = (
    df.groupby("gender")[["dec"] + attributs]
    .corr()
    .loc[(slice(None), "dec"), attributs]   # on ne garde que la ligne 'dec'
    .droplevel(1)
    .round(3)
)
correlations.index = correlations.index.map({0: "Femmes", 1: "Hommes"})
correlations


,attr,sinc,intel,fun,amb,shar
gender,,,,,,
Femmes,0.445,0.222,0.233,0.417,0.175,0.413
Hommes,0.515,0.191,0.217,0.407,0.220,0.387


In [24]:

corr_long = correlations.reset_index().melt(
    id_vars="gender",
    var_name="Attribut",
    value_name="Corrélation avec 'dec'"
)
corr_long = corr_long.rename(columns={"gender": "Genre"})

fig = px.bar(
    corr_long,
    x="Corrélation avec 'dec'",
    y="Attribut",
    color="Genre",
    barmode="group",
    orientation="h",
    title="Poids de chaque attribut dans la décision, par genre",
    text_auto=".2f"
)
fig.update_yaxes(categoryorder="total ascending")
fig.show()


# Ce que les gens disent chercher vs ce qui compte vraiment

In [25]:
prefs = ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"]

# On prend une ligne par participant (les préférences sont les mêmes pour toutes les dates d'une personne)
participants = df.drop_duplicates("iid")[["iid", "gender", "wave"] + prefs].copy()

# Normaliser en pourcentage du total (vaut 100 % pour chaque personne, quelle que soit l'échelle d'origine)
participants[prefs] = participants[prefs].div(participants[prefs].sum(axis=1), axis=0) * 100

# Moyenne par genre
importance_declaree = participants.groupby("gender")[prefs].mean().round(1)
importance_declaree.index = importance_declaree.index.map({0: "Femmes", 1: "Hommes"})
importance_declaree


,attr1_1,sinc1_1,intel1_1,fun1_1,amb1_1,shar1_1
gender,,,,,,
Femmes,18.0,18.2,21.0,17.3,12.8,12.7
Hommes,27.2,16.3,19.4,17.6,8.7,10.9


In [26]:
# 1. Importance déclarée (déjà calculée, on renomme les colonnes pour enlever le suffixe "1_1")
declaree = importance_declaree.copy()
declaree.columns = attributs          # ["attr", "sinc", "intel", "fun", "amb", "shar"]

# 2. Importance réelle = part de chaque corrélation dans la somme des 6
reelle = correlations.div(correlations.sum(axis=1), axis=0) * 100
reelle = reelle.round(1)

# 3. Empiler les deux dans un même DataFrame long
declaree_long = declaree.reset_index().melt(
    id_vars="gender", var_name="Attribut", value_name="Part (%)"
)
declaree_long["Type"] = "Déclarée (signup)"

reelle_long = reelle.reset_index().melt(
    id_vars="gender", var_name="Attribut", value_name="Part (%)"
)
reelle_long["Type"] = "Réelle (corrélation avec 'dec')"

comparaison = pd.concat([declaree_long, reelle_long], ignore_index=True)
comparaison = comparaison.rename(columns={"gender": "Genre"})
comparaison.head(12)


,Genre,Attribut,Part (%),Type
0,Femmes,attr,18.0,Déclarée (signup)
1,Hommes,attr,27.2,Déclarée (signup)
2,Femmes,sinc,18.2,Déclarée (signup)
3,Hommes,sinc,16.3,Déclarée (signup)
4,Femmes,intel,21.0,Déclarée (signup)
5,Hommes,intel,19.4,Déclarée (signup)
6,Femmes,fun,17.3,Déclarée (signup)
7,Hommes,fun,17.6,Déclarée (signup)
8,Femmes,amb,12.8,Déclarée (signup)
9,Hommes,amb,8.7,Déclarée (signup)


In [27]:
fig = px.bar(
    comparaison,
    x="Attribut",
    y="Part (%)",
    color="Type",
    barmode="group",
    facet_col="Genre",
    title="Importance des attributs : ce que les gens disent chercher vs ce qui compte vraiment",
    text_auto=".1f"
)
fig.show()


**Ce que les gens disent chercher vs ce qui compte vraiment**

Les participants surestiment l'importance de la **sincérité** et de l'**intelligence** au moment du signup : ils déclarent y accorder ~18-21 % de leur attention, mais ces attributs ne pèsent en réalité que ~10 % de la décision finale. Ce sont des **qualités attendues comme prérequis**, pas des critères discriminants.

À l'inverse, les **intérêts partagés** (`shar`) sont **sous-déclarés** : cotés à seulement ~12 % au signup, ils représentent en réalité ~20 % du poids de la décision — autant que l'attractivité chez les femmes.

Les **hommes sont cohérents avec eux-mêmes sur l'attractivité** (déclaré 27 %, réel ~23 %), tandis que les **femmes sous-déclarent** ce critère (18 % déclaré, ~22 % réel).

→ **Interprétation** : les profils ne devraient pas chercher à prouver leur intelligence ou sincérité (ces qualités n'influencent presque pas le swipe). Mieux vaut mettre en avant des **centres d'intérêt concrets** — c'est un levier sous-estimé par les utilisateurs eux-mêmes.


## Taux de match selon 'samerace'

In [28]:
taux_samerace = df.groupby("samerace")["match"].mean().round(3)
taux_samerace.index = taux_samerace.index.map({0: "Races différentes", 1: "Même race"})
taux_samerace


samerace
Races différentes    0.161
Même race            0.171
Name: match, dtype: float64

-> partager la même ethnie n'a quasiment pas d'effet sur le taux de match.

## Taux de match selon int_corr

In [29]:
# Découpage en 4 tranches (quartiles)
df["int_corr_tranche"] = pd.qcut(
    df["int_corr"],
    q=4,
    labels=["Très faible", "Faible", "Élevée", "Très élevée"]
)

taux_intcorr = df.groupby("int_corr_tranche", observed=True)["match"].mean().round(3)
taux_intcorr


int_corr_tranche
Très faible    0.153
Faible         0.153
Élevée         0.166
Très élevée    0.186
Name: match, dtype: float64

In [30]:
# Créer un DataFrame unifié pour les deux variables
effet_samerace = pd.DataFrame({
    "Critère": "Même race",
    "Niveau": ["Non", "Oui"],
    "Taux de match": [taux_samerace.iloc[0], taux_samerace.iloc[1]]
})

effet_intcorr = pd.DataFrame({
    "Critère": "Intérêts partagés",
    "Niveau": taux_intcorr.index.astype(str),
    "Taux de match": taux_intcorr.values
})

effets = pd.concat([effet_samerace, effet_intcorr], ignore_index=True)

fig = px.bar(
    effets,
    x="Niveau",
    y="Taux de match",
    color="Critère",
    facet_col="Critère",
    title="Impact sur le taux de match : même race vs intérêts partagés",
    text_auto=".1%"
)
fig.update_yaxes(matches=None)     # chaque sous-graphique a son propre axe y (mais ici même échelle)
fig.update_xaxes(matches=None)     # idem pour l'axe x (niveaux différents)
fig.show()


**Les intérêts partagés comptent plus que l'origine ethnique**

Partager la même ethnie n'augmente le taux de match que de **1 point de pourcentage** (16,1 % → 17,1 %). L'effet est marginal.

En revanche, un très fort alignement des centres d'intérêt (`int_corr` dans le dernier quartile) fait passer le taux de match de **15,3 % à 18,6 %**, soit un gain de **+3,3 points** — environ **trois fois plus** que l'effet "même race". L'effet n'apparaît cependant qu'au-delà d'un seuil : les deux premières tranches (`int_corr` faible ou moyennement faible) donnent le même taux de match.

→ **Intérptretation** : Etre de même origine est beaucoup moins décisive qu'avoir les même goûts'. Pour Tinder, cela suggère que mettre en avant des intérêts spécifiques est un levier plus puissant pour le matching que les filtres démographiques.


# Auto-perception vs perception par les partenaires

In [31]:
# Note moyenne d'attractivité REÇUE par chaque personne (moyenne des notes que ses partenaires lui ont données)
note_recue = df.groupby("pid")["attr"].mean().round(2)
note_recue.name = "attr_recue"

# On récupère l'auto-évaluation (une ligne par personne via drop_duplicates)
auto_eval = df.drop_duplicates("iid")[["iid", "gender", "attr3_1"]].copy()

# On fusionne : on aligne 'iid' (la personne) avec les notes reçues (où elle est 'pid')
perception = auto_eval.merge(note_recue, left_on="iid", right_index=True, how="inner")
perception["gender"] = perception["gender"].map({0: "Femmes", 1: "Hommes"})

perception.head()


,iid,gender,attr3_1,attr_recue
0,1,Femmes,6.0,6.7
10,2,Femmes,7.0,7.7
20,3,Femmes,8.0,6.5
30,4,Femmes,7.0,7.0
40,5,Femmes,6.0,5.3


In [32]:
fig = px.scatter(
    perception,
    x="attr3_1",
    y="attr_recue",
    color="gender",
    title="Auto-évaluation d'attractivité vs note moyenne reçue",
    labels={
        "attr3_1": "Note que la personne se donne (1-10)",
        "attr_recue": "Note moyenne reçue des partenaires (1-10)",
        "gender": "Genre"
    },
    opacity=0.6,
    trendline="ols"   # droite de régression par groupe (ligne d'ajustement)
)

# Ajouter la diagonale y = x en référence
fig.add_shape(
    type="line",
    x0=0, y0=0, x1=10, y1=10,
    line=dict(color="gray", dash="dash")
)

fig.show()


In [33]:
# Décalage = note que la personne se donne − note reçue
# Positif = surestimation, négatif = sous-estimation
perception["decalage"] = perception["attr3_1"] - perception["attr_recue"]

# Statistiques par genre
stats_decalage = perception.groupby("gender")["decalage"].agg(["mean", "median", "std"]).round(2)
stats_decalage


,mean,median,std
gender,,,
Femmes,0.77,0.77,1.57
Hommes,1.02,0.96,1.51


In [34]:
fig = px.histogram(
    perception,
    x="decalage",
    color="gender",
    barmode="overlay",
    opacity=0.6,
    nbins=30,
    title="Distribution du décalage (auto-évaluation − note reçue), par genre",
    labels={"decalage": "Décalage (>0 = surestimation, <0 = sous-estimation)", "gender": "Genre"}
)

# Ligne verticale à zéro (référence = perception juste)
fig.add_vline(x=0, line_dash="dash", line_color="gray")

fig.show()


**Peut-on s'évaluer correctement sur le marché de la rencontre ?**

Plutôt non. La majorité des participants **se surestiment** en attractivité :
- Les femmes surestiment leur note en moyenne de **+0,77 point** sur 10.
- Les hommes de **+1,02 point** — un tiers plus que les femmes.

Le scatter plot le montre nettement : les droites d'ajustement sont bien plus plates que la diagonale. Les personnes qui se donnent 9-10 reçoivent en réalité 6-7 en moyenne. À l'inverse, celles qui se mettent 2-4 sont mieux perçues que ce qu'elles croient.

L'écart-type autour de 1,5 point indique une vraie diversité : certains sont très lucides, d'autres se trompent beaucoup. Mais globalement, le biais est systématique, et plus fort chez les hommes.

→ **Interpretation** : dans un contexte de swipe, les utilisateurs tendent à se croire plus attirants qu'ils ne le sont, et donc à viser des profils au-dessus de leur portée. Cela contribue au "paradoxe du marché" des apps de rencontre (beaucoup de "likes" envoyés, peu de retours). Un feedback honnête sur la réception des profils serait un vrai service pour les utilisateurs.


# Taux de "oui" en fonction de la position (nombre de dates)

In [35]:
# Taux de 'oui' par position dans la soirée
taux_par_ordre = df.groupby("order")["dec"].mean().round(3)
taux_par_ordre


order
1     0.499
2     0.394
3     0.421
4     0.446
5     0.423
6     0.442
7     0.416
8     0.409
9     0.444
10    0.420
11    0.401
12    0.374
13    0.406
14    0.417
15    0.401
16    0.398
17    0.375
18    0.395
19    0.478
20    0.396
21    0.337
22    0.386
Name: dec, dtype: float64

In [36]:
ordre_df = taux_par_ordre.reset_index()

fig = px.line(
    ordre_df,
    x="order",
    y="dec",
    markers=True,
    title="Taux de 'oui' selon la position dans la soirée",
    labels={"order": "Position dans la soirée", "dec": "Taux de décisions positives"}
)

# Ligne horizontale = taux moyen global, pour référence
moyenne_globale = df["dec"].mean()
fig.add_hline(
    y=moyenne_globale,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Moyenne globale : {moyenne_globale:.1%}"
)

fig.show()


**Être le premier de la soirée est un vrai avantage**

Le **premier speed date** de la soirée bénéficie d'un taux de "oui" de **49,9 %**, soit **8 points au-dessus de la moyenne globale** (~42 %). C'est l'avantage le plus net observé dans toute cette analyse.

Ensuite, le taux oscille entre 37 % et 45 % sans tendance claire pour les positions 2 à 15, puis **décroît légèrement en fin de soirée** (positions 17-22, autour de 37-40 %) — signe d'une possible fatigue décisionnelle ou d'un effet de comparaison avec les dates précédents.

→ **Interpretation** : à l'ouverture d'une soirée (ou d'une session de swipe), les utilisateurs sont plus ouverts et moins critiques. En fin de parcours, ils ont accumulé des références et deviennent plus sélectifs. Pour Tinder, cela pourrait suggérer que les **premières minutes d'une session** sont un moment privilégié pour présenter certains profils — ou à l'inverse, que la lassitude pousse à fermer l'app avant que des profils intéressants soient vus.


# Synthèse pour l'équipe marketing de Tinder

**Question initiale :** qu'est-ce qui fait qu'une personne dit "oui" à un second rendez-vous ?

Après analyse de 8 378 rencontres de 4 minutes entre 551 participants, six enseignements clés ressortent.

### 1. Les hommes et les femmes ne décident pas de la même façon

Les hommes disent "oui" 47 % du temps, les femmes 37 %. Toute stratégie de matching doit tenir compte de ces deux seuils d'acceptation très différents.

### 2. Trois attributs déclenchent la décision, trois autres ne discriminent pas

Les **déclencheurs** (corrélation avec la décision ≥ 0,4) sont, pour les deux genres : l'**attractivité**, le **fun** et les **intérêts partagés**. Les **non-déclencheurs** (corrélation ~ 0,2) sont la **sincérité**, l'**intelligence** et l'**ambition** — non parce qu'elles sont mal notées, mais parce qu'elles sont notées élevées pour tout le monde. Ce sont des prérequis attendus, pas des différentiateurs.

### 3. Les gens se trompent sur ce qu'ils cherchent

Au signup, les participants déclarent accorder ~20 % de leur attention à la sincérité et à l'intelligence. En réalité, ces attributs ne pèsent que ~10 % de leur décision finale. À l'inverse, les **intérêts partagés** sont **sous-déclarés** (12 % au signup) alors qu'ils représentent ~20 % du poids réel de la décision.

→ Les utilisateurs ne sont pas les meilleurs juges de leurs propres critères. Leur comportement révèle des préférences différentes de leur discours.

### 4. Les intérêts partagés comptent trois fois plus que l'origine ethnique

Partager la même ethnie n'augmente le taux de match que de **+1 point**. Un fort alignement des centres d'intérêt augmente ce taux de **+3,3 points**. L'homophilie comportementale est nettement plus puissante que l'homophilie culturelle.

### 5. Les gens se surestiment en attractivité, les hommes plus que les femmes

Le décalage moyen entre auto-évaluation et note reçue est de **+0,77 point** chez les femmes et **+1,02 point** chez les hommes. Les personnes qui se mettent 9-10 reçoivent en moyenne 6-7. Biais systématique qui explique en partie le "paradoxe du marché" des apps de rencontre : tout le monde vise plus haut que sa propre ligue perçue.

### 6. Le premier date de la soirée a un net avantage

Le taux de "oui" passe de ~42 % en moyenne à **49,9 % pour la toute première rencontre** de la soirée, puis baisse légèrement en fin de parcours (37-40 % aux dernières positions). Effet d'ouverture en début de session et de fatigue décisionnelle en fin.

---

### Recommandations actionnables pour Tinder

1. **Valoriser les centres d'intérêt dans les profils** (plutôt que de pousser les utilisateurs à "prouver" leur intelligence ou leur sincérité) — levier sous-estimé à fort impact.
2. **Relativiser l'importance des filtres démographiques** (comme l'origine) au profit de filtres comportementaux (goûts, hobbies) — effet trois fois plus puissant sur le match.
3. **Gérer le "biais de surestimation"** : encourager les utilisateurs à swiper aussi sur des profils proches de leur score réel, pas uniquement "au-dessus". Un feedback sur la réception de leur profil pourrait améliorer la qualité des matchs.
4. **Exploiter les premières minutes d'une session** : c'est là que l'utilisateur est le plus ouvert. Les profils présentés en début de session ont une probabilité de like mécaniquement plus élevée.
5. **Segmenter les stratégies par genre** : les hommes sont nettement plus sensibles à l'attractivité (corrélation 0,52), les femmes plus équilibrées entre attractivité, fun et intérêts partagés. La mise en avant des profils peut être adaptée en conséquence.
